# Section 2.1 — Schrödinger Equation for a Central Potential (Weinberg §2.1)

**What this section claims**
- For V = V(r) the Hamiltonian H = p²/2μ + V(r) commutes with the orbital angular momentum L = x × p, so states are labeled simultaneously by (E, l, m): eigenvalues of H, L², L₃.
- Components obey [Lᵢ, Lⱼ] = iℏ εᵢⱼₖ Lₖ; the spectrum of L² is derived **algebraically** (ladder operators) *before* any coordinate representation appears: L² → ℏ²l(l+1), L₃ → ℏm, m = −l, …, +l.
- Separation ψ = R(r)Y(θ,φ) with u = rR turns the Schrödinger equation into a 1D radial problem with a centrifugal barrier ℏ²l(l+1)/(2μr²).
- Boundary conditions u(0) = 0, u(∞) → 0; fixed E is (2l+1)-fold degenerate in m (rotational symmetry).


## [Lᵢ, Lⱼ] = iℏ εᵢⱼₖ Lₖ and [H, Lᵢ] = 0 — verified on a test function (symbolic)

Act on a generic f(x, y, z) with the differential operators Lᵢ = −iℏ εᵢⱼₖ xⱼ ∂ₖ; the commutator must return iℏ εᵢⱼₖ Lₖ f for all nine pairs. For [H, L₃]: the kinetic part commutes because ∇² is rotationally invariant, and the potential part drops out because x∂y − y∂x kills any function of r alone.


In [ ]:
import sympy as sp

x, y, z, hbar, mu = sp.symbols('x y z hbar mu', positive=True, real=True)
f = sp.Function('f')
u = f(x, y, z)

def L(g, i):
    # L_i = -i*hbar*eps_ijk x_j d_k  (1,2,3 = x,y,z)
    if i == 1: return -sp.I*hbar*(y*sp.diff(g, z) - z*sp.diff(g, y))
    if i == 2: return -sp.I*hbar*(z*sp.diff(g, x) - x*sp.diff(g, z))
    return -sp.I*hbar*(x*sp.diff(g, y) - y*sp.diff(g, x))

eps = {(1,2,3):1,(2,3,1):1,(3,1,2):1,(2,1,3):-1,(1,3,2):-1,(3,2,1):-1}
ok = True
for i in (1,2,3):
    for j in (1,2,3):
        lhs = sp.simplify(L(L(u, j), i) - L(L(u, i), j))          # [L_i, L_j] f
        rhs = sp.Integer(0)
        for k in (1,2,3):
            rhs += sp.I*hbar*eps.get((i,j,k), 0)*L(u, k)          # i hbar eps_ijk L_k f
        d = sp.simplify(lhs - rhs)
        ok &= (d == 0)
print("[L_i, L_j] f - i*hbar*eps_ijk L_k f = 0 for all 9 pairs:", ok)

# [p^2, L_3] = 0 : laplacian commutes with the z-rotation generator
lap = lambda g: sp.diff(g, x, 2) + sp.diff(g, y, 2) + sp.diff(g, z, 2)
print("[laplacian, L_3] f = 0:", sp.simplify(lap(L(u,3)) - L(lap(u),3)) == 0)

# [V(r), L_3] = 0 : L_3 is a derivation, so [V, L_3]f = -(L_3 V) f, and L_3 V(r) = 0
r = sp.sqrt(x**2 + y**2 + z**2)
V = sp.Function('V')(r)
L3V = -sp.I*hbar*(x*sp.diff(V, y) - y*sp.diff(V, x))
print("L_3 V(r) = 0:", sp.simplify(L3V) == 0, " -> [H, L_3] = 0 for any central V")

## Ladder algebra → spectrum L² = ℏ²l(l+1), m = −l,…,+l (algebraic)

From [L₁, L₂] = iℏL₃ cyclically, defining L± ≡ L₁ ± iL₂:
- [L₃, L±] = ±ℏ L± — L± raises/lowers m by one unit of ℏ.
- L² = L₃² + ℏL₃ + L₋L₊ (expand L₋L₊ = L₁² + L₂² + i[L₂, L₁] = L₁² + L₂² + ℏL₃).
- L₋L₊ is positive semidefinite, so ℏ²m² ≤ eigenvalue of L²; a highest m = l must exist where L₊ annihilates: L² = ℏ²l(l+1). Stepping down 2l times must reach −l in integer steps ⇒ l integer, m = −l,…,+l, degeneracy 2l+1.
- This is pure algebra — no spherical harmonics needed yet. (Weinberg returns to this machinery for arbitrary j in §4.2.)
Verified explicitly below in the l = 1 matrix representation (Lᵢ)ⱼₖ = −iℏ εᵢⱼₖ:


In [ ]:
import numpy as np

# (L_i)_{jk} = -i eps_{ijk}  -- built programmatically, no hand transcription
eps = np.zeros((3,3,3))
eps[0,1,2]=eps[1,2,0]=eps[2,0,1]=1
eps[0,2,1]=eps[2,1,0]=eps[1,0,2]=-1
L = [-1j*eps[i] for i in range(3)]      # 0-indexed: 0 -> L1, 1 -> L2, 2 -> L3

comm_ok = all(np.allclose(L[i]@L[j]-L[j]@L[i], 1j*sum(eps[i,j,k]*L[k] for k in range(3)))
              for i in range(3) for j in range(3))
print("matrix commutation [L_i, L_j] = i eps_ijk L_k :", comm_ok)

Lsq = sum(Li@Li for Li in L)
print("L^2 = 2 I (i.e. hbar^2 l(l+1), l=1):", np.allclose(Lsq, 2*np.eye(3)))

# algebraic identity L^2 = L3^2 + L3 + L- L+  (hbar = 1)
Lp, Lm_ = L[0] + 1j*L[1], L[0] - 1j*L[1]
print("L^2 = L3^2 + L3 + L- L+ :", np.allclose(Lsq, L[2]@L[2] + L[2] + Lm_@Lp))

# L3 eigenvalues must be -1, 0, +1 in this (l=1) representation
print("L3 eigenvalues:", np.round(np.linalg.eigvalsh(L[2]), 6))

## Separation ψ = u(r)Y(θ,φ)/r — radial equation with centrifugal barrier (symbolic)

With ∇²(RY) = Y(R″ + 2R′/r) − R L²Y/(ℏ²r²) and L²Y = ℏ²l(l+1)Y, setting R = u/r collapses the radial derivative to u″/r:


In [ ]:
r = sp.symbols('r', positive=True)
l = sp.symbols('l', positive=True, integer=True)
u_ = sp.Function('u')
R = u_(r)/r
radial_part = sp.diff(R, r, 2) + 2*sp.diff(R, r)/r
target = sp.Derivative(u_(r), r, 2)/r
print("R'' + 2R'/r with R = u/r simplifies to u''/r :",
      sp.simplify(radial_part - target) == 0)

# hence radial equation:  -(hbar^2/2mu) u'' + [V + hbar^2 l(l+1)/(2 mu r^2)] u = E u
print()
print("radial eq: -(hbar^2/2mu) u'' + [V(r) + hbar^2 l(l+1)/(2 mu r^2)] u = E u")
print("boundary conditions: u(0) = 0  (u ~ r^{l+1}),  u -> 0 at infinity (normalizable)")

## Numerics: effective potential and the centrifugal barrier


In [ ]:
import numpy as np, matplotlib.pyplot as plt, sys
sys.path.insert(0, '..')
from weinberg_constants import e, hbar, me, Ryd, a0

r = np.linspace(1e-2*a0, 12*a0, 800)
mu = me
fig, ax = plt.subplots(figsize=(7, 4.2))
for l in range(4):
    Veff = -e**2/r + hbar**2*l*(l+1)/(2*mu*r**2)
    ax.plot(r/a0, Veff/Ryd, label=f'l = {l}')
ax.axhline(-1.0, color='k', ls=':', lw=1)
ax.text(11.6, -0.97, 'E_1 = -Ryd (hydrogen ground state)', ha='right', fontsize=8)
ax.set_xlabel('r / a0'); ax.set_ylabel('V_eff / Ryd')
ax.set_title('Coulomb + centrifugal barrier:  l >= 1 needs tunneling to reach the origin')
ax.set_ylim(-1.2, 2.0); ax.legend(); plt.tight_layout(); plt.show()

# minimum of V_eff (classical equilibrium radius) for each l
rmin = np.array([hbar**2*l*(l+1)/(mu*e**2) for l in range(1, 5)])
print("location of V_eff minimum, r_min = l(l+1) a0 :", np.round(rmin/a0, 2), "(in units of a0)")

## My summary
*(fill in — own words, ≤ 5 sentences)*

## Gaps
*(questions to raise in review)*
